In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import random
import ast

plt.style.use("seaborn")

## Loading the data & Micro exploration
<hr/>

### Objectives 
* Find out column Dtypes
* Estimate missing values in each column
* Locate and analyze categorical data/columns

In [ ]:
data_frame = pd.read_csv("/kaggle/input/netflix-tv-shows-and-movies/titles.csv")
data_frame.head()

In [ ]:
data_frame.info()

In [ ]:
data_frame.isna().sum()

In [ ]:
isna_result = data_frame.isna().sum().to_dict()
print("Total # of Anomalies (missing values) found : ", sum([val for val in isna_result.values()]))

In [ ]:
categorical_cols = {
    'type': 0, 
    'release_year': 0, 
    'age_certification': 0
    # not including genre yet as it requires fixing 
}

for col_ in categorical_cols.keys():
    categorical_cols[col_] += len(data_frame[col_].value_counts().to_dict().keys())

print(categorical_cols)

### {'category_type': 'categories that exist within that type'}

In [ ]:
data_frame.sort_values(['release_year', 'imdb_score'], ascending=False)

<p>Judging from the histogram below it seems that movies within this dataset started <b>small</b> around the <i>1950s</i> but then drastically <b>increases</b> in the early <i>2000s</i> and onwards</p>

In [ ]:
data_frame["release_year"].hist(figsize=(10, 5));

> Regarding the age certifications for the films its pretty clear that **TV-MA (Mature Audience)** rating the greatest amount of movies under it

In [ ]:
data_frame['age_certification'].hist(figsize=(10, 5));

In [ ]:
grouped_df = data_frame.groupby(["release_year"]).mean()
grouped_df

## Cleansing the data
<hr/>

### Objectives 
* Unpack and repair the **production_countries** and **genre** column values which are currently arrays
* Apply reasonable strategies to drop columns and/or rows with missing data
* Correct column data types where needed 

In [ ]:
def repair_array_bound_categories(arr):
    arr = ast.literal_eval(arr)
    
    if len(arr) == 0:
        return np.nan
    
    elif len(arr) == 1:
        return arr[0]
    
    else:
        return random.choice(arr)
        # return ":".join(arr)

> * Instead of joining together the values which were in an array into a string with a separator we can select a **random** category to be the one for a record
> * Either way its up to you regarding which strategy to use
> * Or another strategy could be you parsing the arrays and then linking them back to the records or perform serialization (I could go on...)

In [ ]:
data_frame["production_countries"] = data_frame["production_countries"].apply(repair_array_bound_categories)
data_frame["genres"] = data_frame["genres"].apply(repair_array_bound_categories)

In [ ]:
data_frame.head()

In [ ]:
columns_to_fill = ("imdb_score", "tmdb_score", "tmdb_popularity")

for col_ in columns_to_fill:
    data_frame[col_].fillna(0.0, inplace=True)

In [ ]:
data_frame.dropna(axis=0, subset=[
    "imdb_votes", 
    "imdb_id", 
    "age_certification", 
    "production_countries",
    "genres",
    "seasons",
    "description"
], inplace=True)

In [ ]:
data_frame["seasons"] = data_frame["seasons"].apply(int)
data_frame["imdb_votes"] = data_frame["imdb_votes"].apply(int)

In [ ]:
data_frame.isna().sum(), data_frame.info()

In [ ]:
data_frame.head()

In [ ]:
data_frame.info()

## Performing analysis on our cleansed data

In [ ]:
ax = sns.kdeplot(data=data_frame["runtime"], shade=True)
ax

In [ ]:
plt.plot(grouped_df.index[17:], grouped_df['imdb_score'][17:], label="IMDB Score Avg", linewidth=2, marker='o', markersize=7)
plt.plot(grouped_df.index[17:], grouped_df["tmdb_score"][17:], label="TMDB Score Avg", linewidth=2, marker='o', markersize=7, linestyle="--")
plt.xlabel("Year")
plt.ylabel("Average")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8), dpi=85)
sns.scatterplot("imdb_score", "tmdb_score", data=data_frame, hue="genres", palette="husl", alpha=0.6);

> * From the distribution plot below its clear to assume that drama, crime and war had an imdb_score that was >= 7

In [ ]:
sns.set(style="darkgrid")
sns.displot(data=data_frame, x='imdb_score', bins=13, hue="genres", alpha=0.4, palette="Dark2");
sns.displot(data=data_frame, x='tmdb_score', bins=13, hue="genres", alpha=0.4)

> * The bar graph below confirms that a great number of movies/shows were produced in the U.S

In [ ]:
plt.figure(figsize=(20, 4), dpi=70)
sns.countplot(data=data_frame, x="production_countries")

> * Thriller, Drama, Fantasy and Crime genres were most active in the TV-MA rating

In [ ]:
plt.figure(figsize=(24, 12), dpi=100)
sns.countplot(data=data_frame, x="age_certification", hue="genres")

In [ ]:
sns.jointplot(x="imdb_score", y="tmdb_score", data=data_frame, hue="age_certification", palette="rainbow", kind="scatter")